<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/snap_mlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.chdir('/content')
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -e .
os.chdir('/content')

In [ ]:
pip install maml numpy==1.23.5

In [ ]:
%%time
# Install LAMMPS with Python interface.
!apt-get update
!apt install -y cmake build-essential git ccache openmpi-bin libopenmpi-dev python3.10-venv
!pip install --upgrade pip
!pip install numpy torch scipy virtualenv psutil pandas tabulate mpi4py Cython sklearn
!pip install ase
# !pip install fitsnap3
%cd /content
!rm -rf lammps
!git clone https://github.com/lammps/lammps.git lammps
%cd /content/lammps
!rm -rf build
!mkdir build
%cd build
!cmake ../cmake -DLAMMPS_EXCEPTIONS=yes \
               -DBUILD_SHARED_LIBS=yes \
               -DMLIAP_ENABLE_PYTHON=yes \
               -DPKG_PYTHON=yes \
               -DPKG_ML-SNAP=yes \
               -DPKG_ML-IAP=yes \
               -DPKG_ML-PACE=yes \
               -DPKG_SPIN=yes \
               -DPYTHON_EXECUTABLE:FILEPATH=`which python`
!make -j 2
!make install-python


# # Install FitSNAP.

# %cd /content
# !rm -rf FitSNAP
# !git clone https://github.com/FitSNAP/FitSNAP
# #!git clone -b hackathon https://github.com/rohskopf/FitSNAP

# # Set environment variables.

# !$PYTHONPATH
# %env PYTHONPATH=/env/python:/bin/bash:
# %env LD_LIBRARY_PATH=/usr/local/nvidia/lib:/usr/local/nvidia/lib64:/content/lammps/build

# # Move into FitSNAP directory
# %cd FitSNAP

In [ ]:
!cp /content/lammps/build/lmp /content/lammps/build/lmp_serial
!chmod 755 /content/lammps/build/lmp_serial


In [ ]:
path = %env PATH
%env PATH=/content/lammps/build/:$path


In [ ]:
!which lmp_serial

In [ ]:
import maml

In [ ]:
from maml.utils import pool_from, convert_docs



In [ ]:
import os
os.chdir('/content')
!wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip

In [ ]:
%%time
import os,json,torch
from jarvis.core.atoms import Atoms
from jarvis.db.jsonutils import loadjson, dumpjson
import json,zipfile
import zipfile
import json
import glob
import pandas as pd
import numpy as np
from jarvis.core.atoms import Atoms
import os
import torch
from jarvis.db.figshare import data
import subprocess
from subprocess import Popen, PIPE
from maml.base import SKLModel
from maml.describers import BispectrumCoefficients
from sklearn.linear_model import LinearRegression
from maml.apps.pes import SNAPotential

mlearn = json.loads(
        zipfile.ZipFile("/content/mlearn.json.zip").read(
            "mlearn.json"
        )
    )

run_dir='/content'
elements = ["Cu","Ni","Ge","Mo","Li","Si"] #["Si"] #,"Ni","Ge","Mo","Li"]
mem = []
for element in elements:
    os.chdir(run_dir)
    dir_name = "maml_snap_" + element
    cmd='rm -rf '+dir_name
    os.system(cmd)
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)
    benchmark_energies = (
        "jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/mlearn_"
        + element
        + "_energy.json.zip"
    )

    temp_energies = benchmark_energies.split("/")[-1].split(".zip")[0]
    energies = json.loads(
        zipfile.ZipFile(benchmark_energies).read(temp_energies)
    )
    train_ids = list(energies["train"].keys())
    test_ids = list(energies["test"].keys())
    train_energies = []
    train_forces = []
    train_stresses = []
    train_structures = []
    for i in mlearn:
        if i["jid"] in train_ids:
            # print(i)
            train_energies.append(i["energy"])
            train_forces.append(i["forces"])
            train_stresses.append(i["stresses"])
            atoms = Atoms.from_dict(i["atoms"])
            train_structures.append(atoms.pymatgen_converter())
               # Val same as test
    test_energies = []
    test_forces = []
    test_stresses = []
    test_structures = []
    tids = []
    for i in mlearn:
        if i["jid"] in test_ids:
            # print(i)
            test_energies.append(i["energy"])
            test_forces.append(i["forces"])
            test_stresses.append(i["stresses"])
            atoms = Atoms.from_dict(i["atoms"])
            test_structures.append(atoms.pymatgen_converter())
            tids.append(i['jid'])
    train_pool = pool_from(train_structures, train_energies, train_forces)

    _, df = convert_docs(train_pool)

    weights = np.ones(len(df['dtype']), )

    weights[df['dtype'] == 'energy'] = 100

    element_profile = {element: {'r': 5.0, 'w': 1}}
    describer = BispectrumCoefficients(rcutfac=0.5, twojmax=6, element_profile=element_profile,
                                      quadratic=False, pot_fit=True)
    model = SKLModel(describer=describer, model=LinearRegression())
    snap = SNAPotential(model=model)
    snap.train(train_structures, train_energies, train_forces, sample_weight=weights)
    ########################################
    def get_snap_forces(snap_model='',test_structure=''):
        test_energy=-9999
        test_force=np.zeros((atoms.num_atoms,3))
        df_orig, df_predict = snap_model.evaluate(test_structures=[test_structure],
                                        test_energies=[test_energy],
                                        test_forces=[test_force], test_stresses=None
                                        )
        return df_predict[df_predict['dtype']=='energy']['y_orig'].values[0],df_predict[df_predict['dtype']=='force']['y_orig'].values,0

    # df = pd.DataFrame(mdata)
    df = pd.DataFrame(
        json.loads(
            zipfile.ZipFile("mlearn.json.zip").read(
                "mlearn.json"
            )
        )
    )
    print(df)
    #for i in glob.glob("../../benchmarks/AI/MLFF/*energy*.zip"):
    for i in glob.glob("/content/jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/*energy*.zip"):
        if "mlearn" in i and element in i:
            fname_e = (
                "AI-MLFF-energy-"
                + i.split("/")[-1].split("_energy.json.zip")[0]
                + "-test-mae.csv"
            )
            fname_f = (
                "AI-MLFF-forces-"
                + i.split("/")[-1].split("_energy.json.zip")[0]
                + "-test-multimae.csv"
            )
            # fname_s = (
            #     "AI-MLFF-stresses-"
            #     + i.split("/")[-1].split("_energy.json.zip")[0]
            #     + "-test-multimae.csv"
            # )
            f_e = open(fname_e, "w")
            f_f = open(fname_f, "w")
            # f_s = open(fname_s, "w")

            f_e.write("id,prediction\n")
            f_f.write("id,prediction\n")
            # f_s.write("id,prediction\n")

            print(i)
            dat = json.loads(
                zipfile.ZipFile(i).read(i.split("/")[-1].split(".zip")[0])
            )
            print(dat["test"])
            for key, val in dat["test"].items():
                entry = df[df["jid"] == key]
                atoms = Atoms.from_dict(entry.atoms.values[0])
                # print(key,val,df[df['jid']==key],atoms)
                # energy,forces=get_alignn_forces(atoms)
                energy, forces, stress = get_snap_forces(snap_model=snap,test_structure=atoms.pymatgen_converter())
                print(key, val, energy, atoms.num_atoms)
                line = key + "," + str(energy) + "\n"
                f_e.write(line)
                line = (
                    key
                    + ","
                    + str(";".join(map(str, np.array(forces).flatten())))
                    + "\n"
                )
                f_f.write(line)
                # line = (
                #     key
                #     + ","
                #     + str(";".join(map(str, np.array(stress).flatten())))
                #     + "\n"
                # )
                # f_s.write(line)
            f_e.close()
            f_f.close()
            # f_s.close()
            zname = fname_e + ".zip"
            with zipfile.ZipFile(zname, "w") as myzip:
                myzip.write(fname_e)

            zname = fname_f + ".zip"
            with zipfile.ZipFile(zname, "w") as myzip:
                myzip.write(fname_f)

            # zname = fname_s + ".zip"
            # with zipfile.ZipFile(zname, "w") as myzip:
            #     myzip.write(fname_s)


